# Competitor Rate Ingestion

This notebook walks through the competitor-rate workflow for the AI Bank Pricing Engine.

The workflow fetches official NZ bank pages, extracts advertised home-loan and term-deposit rates, validates the extracted records, stores audit artifacts, and can apply selected market reference rates to the pricing engine.

This project uses synthetic data only. The competitor rates are scenario inputs, not financial advice.

## How To Use This Notebook

Run the cells from top to bottom.

Live fetching reaches out to official bank websites, so it is disabled by default. When you intentionally want to refresh live artifacts, set `RUN_LIVE_FETCH = True` in the fetch cell.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Open this notebook from the repository root or the docs directory.")

os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

## Official Sources

The configured official sources cover ANZ, ASB, Westpac, BNZ, and Kiwibank for:

- Home loans and mortgages
- Ordinary retail term deposits
- PIE term deposits where they appear on the same official investment-rate pages

Third-party comparison pages are not used to override official sources.

In [ ]:
import pandas as pd

from balance_pricing_engine.competitor_rates.sources import OFFICIAL_SOURCES

sources = pd.DataFrame(
    [
        {
            "bank": source.bank,
            "product_type": source.product_type.value,
            "url": source.url,
        }
        for source in OFFICIAL_SOURCES
    ]
)
sources

## Fetch Official Pages

Fetching writes raw HTML snapshots plus processed JSON, CSV, history, and validation-report artifacts.

The browser fallback is useful when a bank page requires JavaScript-rendered values and Playwright is installed. It is opt-in so the default workflow does not add a browser dependency.

In [ ]:
import subprocess
import sys

RUN_LIVE_FETCH = True
FETCH_SOURCE = "interest_co_nz"  # use "official" to try bank websites directly
USE_BROWSER_FALLBACK = False

if FETCH_SOURCE == "interest_co_nz":
    cmd = [
        sys.executable,
        "-m",
        "balance_pricing_engine.competitor_rates.cli",
        "fetch-interest-co-nz",
    ]
elif FETCH_SOURCE == "official":
    cmd = [
        sys.executable,
        "-m",
        "balance_pricing_engine.competitor_rates.cli",
        "fetch",
        "--strict",
        "false",
    ]
else:
    raise ValueError("FETCH_SOURCE must be 'interest_co_nz' or 'official'")

if FETCH_SOURCE == "official" and USE_BROWSER_FALLBACK:
    cmd.append("--browser-fallback")

print("Command:", " ".join(cmd))
if FETCH_SOURCE == "official" and USE_BROWSER_FALLBACK:
    try:
        import playwright  # noqa: F401
    except ImportError:
        print("Browser fallback requested, but Playwright is not installed in this kernel environment.")
        print("Install it with: python -m pip install playwright && python -m playwright install chromium")

if RUN_LIVE_FETCH:
    result = subprocess.run(cmd, check=False)
    if result.returncode:
        print("Live fetch finished with failures. This is common when bank sites block automation, time out, or change their page layout.")
        print("Review the source-level messages above before using the generated artifact.")
else:
    print("Live fetch skipped. Set RUN_LIVE_FETCH = True when you want to refresh live rate artifacts.")

## Artifacts

A fetch run writes:

- `data/raw/competitor_rates/YYYY-MM-DD/<bank>_<product_type>.html`
- `data/processed/competitor_rates/latest.json`
- `data/processed/competitor_rates/latest.csv`
- `data/processed/competitor_rates/history/YYYY-MM-DDTHH-MM-SSZ.json`
- `data/processed/competitor_rates/validation_report_latest.json`

Raw and processed live artifacts are gitignored. Tests use sanitised fixtures under `tests/fixtures/competitor_rates/`.

In [ ]:
processed_dir = PROJECT_ROOT / "data" / "processed" / "competitor_rates"

if processed_dir.exists():
    for path in sorted(processed_dir.glob("*")):
        print(path)
else:
    print(f"No processed competitor-rate directory found yet: {processed_dir}")

## Validate An Existing Artifact

Validation checks required fields, numeric rates, expected rate ranges, duplicates, missing sources, parser failures, large historical movements, and staleness.

In [ ]:
from datetime import datetime, timezone

from balance_pricing_engine.competitor_rates.store import load_rate_run

USE_DEMO_FIXTURE = False

live_source_file = PROJECT_ROOT / "data" / "processed" / "competitor_rates" / "latest.json"
demo_source_file = PROJECT_ROOT / "data" / "processed" / "competitor_rates" / "demo_latest.json"
fixture_source_file = PROJECT_ROOT / "tests" / "fixtures" / "competitor_rates" / "latest.json"

if USE_DEMO_FIXTURE:
    now = datetime.now(timezone.utc)
    fixture_run = load_rate_run(fixture_source_file)
    refreshed_records = [
        record.model_copy(update={"captured_at_utc": now})
        for record in fixture_run.records
    ]
    demo_run = fixture_run.model_copy(
        update={"captured_at_utc": now, "records": refreshed_records}
    )
    demo_source_file.parent.mkdir(parents=True, exist_ok=True)
    demo_source_file.write_text(demo_run.model_dump_json(indent=2), encoding="utf-8")
    source_file = demo_source_file
    print(f"Using sanitized demo fixture: {source_file}")
else:
    source_file = live_source_file
    print(f"Using live fetch artifact: {source_file}")

def competitor_rate_file_is_usable(path):
    if not path.exists():
        return False
    run = load_rate_run(path)
    return bool(run.records) and run.validation_summary.status != "failed"

cmd = [
    sys.executable,
    "-m",
    "balance_pricing_engine.competitor_rates.cli",
    "validate",
    "--source-file",
    str(source_file),
    "--strict",
    "false",
]
print("Command:", " ".join(cmd))

if source_file.exists():
    result = subprocess.run(cmd, check=False)
    if result.returncode:
        print("Validation found hard failures. Keep the artifact for diagnosis, but do not use it as a pricing input.")
else:
    print(f"Nothing to validate yet. Missing file: {source_file}")

## Print A Stored Run Summary

Use the report command for a compact summary of the latest stored run.

In [ ]:
cmd = [
    sys.executable,
    "-m",
    "balance_pricing_engine.competitor_rates.cli",
    "report",
    "--source-file",
    str(source_file),
]
print("Command:", " ".join(cmd))

if source_file.exists():
    subprocess.run(cmd, check=True)
else:
    print(f"Nothing to report yet. Missing file: {source_file}")

## Inspect Extracted Records

The stored run can also be loaded directly in Python for review.

In [ ]:
from balance_pricing_engine.competitor_rates.store import load_rate_run

if source_file.exists():
    run = load_rate_run(source_file)
    records = pd.DataFrame([record.model_dump(mode="json") for record in run.records])
    display(records.head(20))
    print(f"Total records: {len(records)}")
    print(f"Validation status: {run.validation_summary.status}")
else:
    print(f"No stored run available yet: {source_file}")

## Normalisation

The parser preserves each advertised term label and derives structured fields where possible.

- `6 months` and `six months` become 6 normalised months.
- `1 year` becomes 12 normalised months.
- `30 days`, `60 days`, and `90 days` retain `term_days_raw` and approximate months for strategy selection.
- `floating` and `variable` remain special labels without a month term.

Interest payment frequency maps to `at_maturity`, `monthly`, `quarterly`, `annually`, or `unknown`.

## Apply Competitor Rates To The Engine Config

Manual config remains supported. If `competitor_rates.enabled=false`, the engine uses the existing `market.competitor_loan_rate`, `market.competitor_deposit_rate`, `market.home_loan_reference_rate`, and `market.term_deposit_reference_rate`.

When enabled, rates are filtered by selected banks, product, term, mortgage conditions, and deposit frequency preference. Supported strategies are:

- Mortgages: `bank_specific`, `market_average`, `market_min`, `market_median`
- Term deposits: `bank_specific`, `market_average`, `market_max`, `market_median`

Resolved advertised percentages are converted to decimal rates before being applied to the engine.

In [ ]:
config_file = PROJECT_ROOT / "configs" / "base.yaml"

cmd = [
    sys.executable,
    "-m",
    "balance_pricing_engine.competitor_rates.cli",
    "apply",
    "--config",
    str(config_file),
    "--source-file",
    str(source_file),
]
print("Command:", " ".join(cmd))

if source_file.exists() and competitor_rate_file_is_usable(source_file):
    subprocess.run(cmd, check=True)
elif source_file.exists():
    print("Cannot apply competitor rates: the source file has no usable validated rate records.")
else:
    print(f"Cannot apply competitor rates yet. Missing file: {source_file}")

In [ ]:
WRITE_GENERATED_CONFIG = False

cmd = [
    sys.executable,
    "-m",
    "balance_pricing_engine.competitor_rates.cli",
    "apply",
    "--config",
    str(config_file),
    "--source-file",
    str(source_file),
    "--write",
]
print("Command:", " ".join(cmd))

if WRITE_GENERATED_CONFIG and source_file.exists() and competitor_rate_file_is_usable(source_file):
    subprocess.run(cmd, check=True)
elif not source_file.exists():
    print(f"Cannot write generated config yet. Missing file: {source_file}")
elif not competitor_rate_file_is_usable(source_file):
    print("Generated config write skipped because the source file has no usable validated rate records.")
else:
    print("Generated config write skipped. Set WRITE_GENERATED_CONFIG = True to create one.")

## Run The Baseline Scenario With Competitor Rates

This runs the pricing scenario after resolving the competitor-rate file into the configured market assumptions.

In [ ]:
cmd = [
    sys.executable,
    "examples/run_with_competitor_rates.py",
    "--source-file",
    str(source_file),
]
print("Command:", " ".join(cmd))

if source_file.exists() and competitor_rate_file_is_usable(source_file):
    subprocess.run(cmd, check=True)
elif source_file.exists():
    print("Cannot run scenario: the source file has no usable validated rate records.")
else:
    print(f"Cannot run scenario yet. Missing file: {source_file}")

## Validation Details

Every run produces a validation summary. The validator checks:

- Required bank, product type, source URL, capture timestamp, and term/product identity
- Numeric percentage rates
- Rates outside the expected 0% to 15% range
- Duplicate records for the same bank/product/term/category/frequency
- Missing official bank/product sources
- Parser failures where a page was fetched but no rates were extracted
- Large historical movements above 0.50 percentage points when a previous run is supplied
- Staleness through `max_staleness_hours`

Strict mode fails when required official sources are missing or failed. Non-strict mode reports warnings and continues with available validated rates.

## Known Limitations

- Bank page layouts can change and may require parser updates.
- Special mortgage rates often have eligibility conditions that must be reviewed.
- PIE term deposits are not the same as ordinary retail term deposits.
- Low-equity mortgage margins are stored separately and are not applied unless a scenario explicitly models them.
- This project uses synthetic data and is not financial advice. Rates must be verified before business use.